In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [6]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [7]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your operating system.
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. Open a terminal and run:
```bash
ollama run llama3
```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

3. To check that the local server is running, you can use:
```bash
curl http://localhost:11434
```

You should get a response like:
```json
{"models": [...]}  
```

4. If you want to use it from Python, install the client with:
```bash
pip install ollama
```


In [8]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I don’t see any information in the context about how to run **Ollama locally**.

The closest related guidance is that you can run the course locally if you’re comfortable setting up the needed tools, but the FAQ does not mention Ollama specifically.


In [9]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Maybe — it depends on the course’s enrollment rules and whether it’s still open.\n\nIf you want, I can help you check the exact policy. Please send me:\n- the course name\n- the school/platform\n- any link or screenshot of the course page\n\nIf you’re asking generally, the usual possibilities are:\n- **Open enrollment:** yes, you can join anytime\n- **Cohort-based course:** only if the new session hasn’t started or if the instructor allows late entry\n- **Limited-capacity course:** only if there’s a seat available\n- **Expired/archived course:** usually no, unless there’s an upcoming run\n\nIf you’d like, I can also help you draft a short message to the instructor asking to join.'

In [10]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [11]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [12]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [13]:
len(response.output)

1

In [14]:
call = response.output[0]

In [15]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join enrollment late registration eligibility"}', call_id='call_3NinzsXMHlvYBfpLJqDutpF1', name='search', type='function_call', id='fc_0750d7decd0ae428006a3989b1b3b481a1a3e178ce810b4ad5', namespace=None, status='completed')

In [16]:
import json

args = json.loads(call.arguments)
args

{'query': 'join course discovered can I join enrollment late registration eligibility'}

In [17]:
call.name

'search'

In [18]:
results = search(**args)

In [19]:
result_json = json.dumps(results, indent=2)

In [20]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [21]:
messages.append(call)

In [22]:
messages.append(function_call_output)

In [23]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join enrollment late registration eligibility"}', call_id='call_3NinzsXMHlvYBfpLJqDutpF1', name='search', type='function_call', id='fc_0750d7decd0ae428006a3989b1b3b481a1a3e178ce810b4ad5', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_3NinzsXMHlvYBfpLJqDutpF1',
  'output': '[\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions.",\n    "doc_id": "74eb249bbf"\n  },\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confi

### Asking the model again

In [24]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [25]:
print(response.output_text)

Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open.


### Token usage and cost

In [26]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(776, 31)

In [27]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0001176


In [28]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [29]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [30]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [31]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment"}
function_call: search {"query":"course enrollment late join discovered course"}
function_call: search {"query":"register enroll after course starts can I join"}


In [32]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment"}', call_id='call_dbZtu84bdVWQVgPfXdzGTYOg', name='search', type='function_call', id='fc_0513f87fb6874a37006a3989d8d46081a09476e50df0003bd1', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment late join discovered course"}', call_id='call_ry2dwJVd5aqeXpRtahlRemtf

### The full agent loop

In [33]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course discovered can I join enrollment registration late join FAQ"}
function_call: search {"query":"course discover can I join now FAQ enrollment"}
function_call: search {"query":"late enrollment join course FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

A couple of notes:
- You can start learning and working through the materials whenever you want.
- If you want a certificate, you need to submit your project while submissions are still open.
- If you’re just joining late, you can still follow the lessons and homework, but self-paced participation doesn’t qualify for a certificate.

If you want, I can also help you figure out the best way to start the course now or explain the certificate requirements in more detail. Is there anything else you want to explore?


### Wrapping it in a function

In [34]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

### Encouraging multiple searches

In [35]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [36]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join the course enrollment late join discovered course can I join course FAQ"}
iteration #2...
function_call: search {"query":"certificate project accepting submissions join course still accepting submissions peer-review live cohort self-paced certificate FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

A couple of important notes:
- You can follow the materials even if you discovered it late.
- If you want a certificate, you need to submit your project while submissions are still open.

If you want, I can also explain how to start the course now or what you need for the certificate.


In [37]:
result

'Yes — you can still join the course.\n\nA couple of important notes:\n- You can follow the materials even if you discovered it late.\n- If you want a certificate, you need to submit your project while submissions are still open.\n\nIf you want, I can also explain how to start the course now or what you need for the certificate.'

### Restricting off-topic questions

In [38]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit queen gambit"}
iteration #2...
function_call: search {"query":"chess gambit queen gambit opening"}
iteration #3...
ASSISTANT:
I couldn’t find any course FAQ entry about “queen gambit,” so I can’t answer that from the course materials.

If you meant a course-related term, try sending the exact course topic or context. Are there other areas you want to explore?


In [39]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [40]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [41]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [42]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [43]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [44]:
[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [45]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [46]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [47]:
result = runner.loop(
    prompt='How do I run Olama locally?',
    callback=callback,
)

-> Response received


-> Response received


In [48]:
result.cost

CostInfo(input_cost=Decimal('0.0034065'), output_cost=Decimal('0.0015345'), total_cost=Decimal('0.0049410'))

In [49]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run local install Ollama FAQ"}', call_id='c

In [50]:
result2 = runner.loop(
    prompt='How do I run a different model?',
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [51]:
runner.run();

KeyboardInterrupt: Interrupted by user